# 01 - Data Discovery

**Purpose:** Validate data availability from the Tausi API before committing to any model architecture.

**Questions to answer:**
1. How many KEHC/HCNRB decisions exist for 2015-2023?
2. How many cite the Data Protection Act specifically?
3. What is the field completeness (judges, advocates, citations)?
4. What is the PDF download success rate?
5. Can we achieve 300+ cases with reliable binary outcomes?

**Gate:** If <300 commercial-division cases with published judgments, broaden the scope.

In [ ]:
import asyncio
import sys
sys.path.insert(0, '..')

from src.data.api_client import TausiClient
from configs.settings import settings

print(f"API Base URL: {settings.tausi.base_url}")
print(f"Target court: {settings.data.target_court_code}")
print(f"Year range: {settings.data.filing_year_start}-{settings.data.filing_year_end}")

## 1. Total Case Counts by Year

In [ ]:
async def count_by_year():
    """Query total available decisions per year."""
    async with TausiClient() as client:
        counts = {}
        for year in range(settings.data.filing_year_start, settings.data.filing_year_end + 1):
            count = await client.count_decisions(
                court=settings.data.target_court_code,
                year=year
            )
            counts[year] = count
            print(f"  {year}: {count} decisions")
        return counts

year_counts = await count_by_year()
total = sum(year_counts.values())
print(f"\nTotal KEHC decisions (2015-2023): {total}")

## 2. Data Protection Act Cases

In [ ]:
async def count_dpa_cases():
    """Search for cases citing the Data Protection Act."""
    async with TausiClient() as client:
        dpa_cases = []
        async for case in client.search_decisions("Data Protection Act"):
            dpa_cases.append(case)
        return dpa_cases

dpa_cases = await count_dpa_cases()
print(f"Cases mentioning 'Data Protection Act': {len(dpa_cases)}")
print("\nNote: DPA enacted Nov 2019. Expect <50 published decisions.")

## 3. Field Completeness Audit (100-Case Sample)

In [ ]:
from src.data.validators import TausiDecisionRaw

async def audit_fields(sample_size=100):
    """Check field completeness on a sample of cases."""
    async with TausiClient() as client:
        sample = []
        async for case in client.iter_decisions(
            court=settings.data.target_court_code, year=2022
        ):
            sample.append(TausiDecisionRaw.model_validate(case))
            if len(sample) >= sample_size:
                break
    
    # Compute completeness
    stats = {
        'has_judges': sum(1 for c in sample if c.judges),
        'has_advocates': sum(1 for c in sample if c.advocates),
        'has_citations': sum(1 for c in sample if c.cited_documents),
        'has_casetype': sum(1 for c in sample if c.casetype),
        'has_judgment_date': sum(1 for c in sample if c.judgment_date),
        'has_filing_year': sum(1 for c in sample if c.filing_year),
    }
    
    print(f"Field completeness ({len(sample)} cases from 2022):")
    for field, count in stats.items():
        pct = 100 * count / len(sample)
        print(f"  {field}: {count}/{len(sample)} ({pct:.1f}%)")
    
    return sample, stats

sample, field_stats = await audit_fields()

## 4. PDF Download Test

In [ ]:
from pathlib import Path
import tempfile

async def test_pdf_downloads(cases, max_test=20):
    """Test PDF download success rate on a subsample."""
    async with TausiClient() as client:
        success = 0
        fail = 0
        for case in cases[:max_test]:
            try:
                pdf_url = f"{client.base_url}/decisions{case.frbr_uri}.pdf"
                with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as tmp:
                    await client.download_file(pdf_url, Path(tmp.name))
                    size = Path(tmp.name).stat().st_size
                    if size > 1000:
                        success += 1
                    else:
                        fail += 1
            except Exception as e:
                fail += 1
    
    print(f"PDF download test ({max_test} cases):")
    print(f"  Success: {success} ({100*success/max_test:.0f}%)")
    print(f"  Failed: {fail} ({100*fail/max_test:.0f}%)")

await test_pdf_downloads(sample)

## 5. Scope Decision

Based on the results above, decide whether to:
- **Option A:** Proceed with narrow scope (DPA + commercial injunctions) — only if 300+ cases available
- **Option B (Recommended):** Broaden to all Commercial Division cases at HCNRB
- **Option C:** Broaden further to all KEHC cases

Document the decision and rationale below before proceeding to Phase 1.

In [ ]:
# Document your scope decision here after reviewing the data above
SCOPE_DECISION = {
    "option": "TBD",  # A, B, or C
    "rationale": "TBD",
    "total_cases_available": total,
    "dpa_cases_available": len(dpa_cases),
    "revised_hypothesis": "TBD",
}
print(SCOPE_DECISION)